# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [61]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("E:\E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

<>:11: SyntaxWarning: invalid escape sequence '\E'
<>:11: SyntaxWarning: invalid escape sequence '\E'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_246136\1984545978.py:11: SyntaxWarning: invalid escape sequence '\E'
  Path("E:\E Commerce Dataset.xlsx"),


原始数据：E:\E Commerce Dataset.xlsx
项目输出目录：d:\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [62]:
# 在此写下你的答案：
# 1.每条记录代表一个唯一的电商客户（用户）的完整行为与属性信息
# 2.目标变量是 Churn（客户流失标记）。
# 3.CustomerID 的数值大小不代表任何顺序、等级或量纲，仅用于标识，它的唯一作用是区分不同客户记录，类似于数据库中的主键。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [63]:
def build_quality_report(data):
    report = pd.DataFrame({
        "column": data.columns,                                    # 字段名称
        "dtype": data.dtypes.values,                               # 数据类型
        "missing_count": data.isnull().sum().values,              # 缺失数量
        "missing_pct": (data.isnull().sum() / len(data) * 100).values,  # 缺失比例(%)
        "unique_count": data.nunique().values,                    # 唯一值数量
    })
    report["missing_pct"] = report["missing_pct"].round(2)
    return report

# TODO：生成清洗前质量报告
# quality_before = build_quality_report(raw_df)
# display(quality_before)

### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [64]:
"""
# TODO：完成项目初始审计
# print("完全重复行数：", ...)
# print("CustomerID 重复数量：", ...)
# print(raw_df["Churn"].value_counts())
# print("流失率：", ...)
#
# for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
#     print(f"\n{col}")
#     print(raw_df[col].value_counts())
"""
# 1. 原始数据的完全重复行数
print("完全重复行数: ", raw_df.duplicated().sum())
# 2. CustomerID 重复数量
print("CustomerID 重复数量: ", raw_df["CustomerID"].duplicated().sum())
# 3. Churn 的频数和流失率
print(raw_df["Churn"].value_counts())
print("流失率: ", raw_df["Churn"].mean() * 100, "%")
# 4. 主要类别字段的频数
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数:  0
CustomerID 重复数量:  0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率:  16.838365896980463 %

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [65]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [66]:
# 定义需要处理的列和映射规则
NUMERIC_MISSING_COLS = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp",
    "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount", "DaySinceLastOrder"
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"           # Phone 与 Mobile Phone 同义合并
    },
    "PreferredPaymentMode": {
        "CC": "Credit Card",               # CC 缩写标准化
        "COD": "Cash on Delivery"          # COD 缩写标准化
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"           # Mobile 与 Mobile Phone 同义合并
    }
}


def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数:
        data: 原始用户行为 DataFrame

    返回:
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    logs = []
    
    # TODO: 复制数据，避免覆盖原始数据
    df = data.copy()
    
    # TODO: 删除完全重复行，并记录日志
    before_dedup = len(df)
    df = df.drop_duplicates(keep="first")
    after_dedup = len(df)
    logs.append({
        "step": "删除完全重复行",
        "rule": "drop_duplicates(keep='first')",
        "before_records": before_dedup,
        "after_records": after_dedup,
        "affected_records": before_dedup - after_dedup,
    })
    
    # TODO: 对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            logs.append({
                "step": f"缺失值填补: {col}",
                "rule": f"中位数填补 (median={median_val:.2f})",
                "before_records": len(df),
                "after_records": len(df),
                "affected_records": int(missing_count),
            })
    
    # TODO: 对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for col, mapping in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mapping.items():
            affected = (df[col] == old_val).sum()
            if affected > 0:
                df[col] = df[col].replace(old_val, new_val)
                logs.append({
                    "step": f"类别标准化: {col}",
                    "rule": f"'{old_val}' -> '{new_val}'",
                    "before_records": len(df),
                    "after_records": len(df),
                    "affected_records": int(affected),
                })
    
    # TODO: 将 Churn 和 Complain 转为整数类型
    int_cols = ["Churn", "Complain"]
    for col in int_cols:
        before_dtype = str(df[col].dtype)
        df[col] = df[col].astype(int)
        logs.append({
            "step": f"数据类型转换: {col}",
            "rule": f"{before_dtype} -> int64",
            "before_records": len(df),
            "after_records": len(df),
            "affected_records": len(df),
        })
    
    # TODO: 返回 cleaned_df 与 cleaning_log
    cleaned_df = df.reset_index(drop=True)
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log


# 运行清洗函数
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

### 任务 3：运行清洗函数并查看日志

In [67]:
# TODO：执行清洗
# cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
#
# display(cleaning_log)
# cleaned_df.head()

---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [68]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum()),
    }


# ============================================================
# TODO 1: 构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# ============================================================
tenure_bins = [0, 3, 9, 15, float("inf")]
tenure_labels = ["0-3年", "4-9年", "10-15年", "16年+"]

cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"], 
    bins=tenure_bins, 
    labels=tenure_labels, 
    right=True,
    include_lowest=True
)

# ============================================================
# TODO 2: 新建 IsMobileLogin，移动端为 1，其他设备为 0
# ============================================================
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# ============================================================
# TODO 3: 生成 outlier_report（每行对应一个待检查字段）
# ============================================================
outlier_fields = ["WarehouseToHome", "OrderCount", "CashbackAmount"]

outlier_report = pd.DataFrame([
    {"field": col, **iqr_outlier_summary(cleaned_df[col])}
    for col in outlier_fields
])

display(outlier_report)

,field,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [69]:

business_rule_report = pd.DataFrame({
    "规则": [
        "Tenure < 0（使用时长小于0）",
        "WarehouseToHome < 0（仓库距离小于0）",
        "OrderCount <= 0（订单数小于或等于0）",
        "CashbackAmount < 0（返现金额小于0）"
    ],
    "不合规记录数": [
        (cleaned_df["Tenure"] < 0).sum(),
        (cleaned_df["WarehouseToHome"] < 0).sum(),
        (cleaned_df["OrderCount"] <= 0).sum(),
        (cleaned_df["CashbackAmount"] < 0).sum()
    ]
})

display(business_rule_report)

,规则,不合规记录数
0,Tenure < 0（使用时长小于0）,0
1,WarehouseToHome < 0（仓库距离小于0）,0
2,OrderCount <= 0（订单数小于或等于0）,0
3,CashbackAmount < 0（返现金额小于0）,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [70]:
quality_before = build_quality_report(raw_df)
quality_after = build_quality_report(cleaned_df)

assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)

# 导出文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

1. 本项目发现了哪些数据质量问题？
发现三类问题：① 缺失值：7个数值列共1,856个缺失值，占比4.5%~5.5%；② 类别不一致：存在同义不同名，如"Phone"与"Mobile Phone"、"CC"与"Credit Card"、"COD"与"Cash on Delivery"；③ 候选异常值：OrderCount有703个、CashbackAmount有438个超出IQR上下界。
2. 对缺失值、类别不一致、候选异常值分别采取了什么策略？
缺失值：使用中位数填补，避免均值受异常值影响，且不采用Churn分组填补以防目标泄露；
类别不一致：建立统一映射表进行标准化合并，如将"CC"映射为"Credit Card"；
候选异常值：遵循项目规则"先记录、不盲目删除"，用IQR方法识别并报告，供后续分析参考。
3. 为什么清洗后的数据可以作为第五天分析的输入？
清洗后数据满足：①缺失值全部处理，无空值；②类别标准化，减少噪声；③数据类型正确（Churn、Complain为整型）；④新增TenureGroup和IsMobileLogin两个衍生特征便于分析；⑤业务规则检查全部通过，无不合规记录；⑥处理过程全程留痕，可复现、可追溯。
4. 哪些处理规则仍需要业务人员确认？
中位数填补是否反映业务实际，特别是高Tenure用户的缺失值；
候选异常值（尤其是OrderCount>6、CashbackAmount>272）是否代表真实的高频/高价值用户；
Tenure分箱区间（0-3/4-9/10-15/16+）是否符业务分层逻辑；
"Phone"合并为"Mobile Phone"是否准确（是否存在固话场景）。